# Argumentation abstraite de Dung — sémantiques grounded, preferred, stable

> **Notebook fondationnel** (non-agentique) de la série *Argument_Analysis*.
> Il précède et éclaire l'aperçu de [2-formal §6](Argument_Analysis_Agentic-2-formal.ipynb)
> et l'utilisation du solveur Tweety dans [Tweety-5](../Tweety/Tweety-5-Abstract-Argumentation.ipynb).

## Pourquoi ce notebook

Le notebook [2-formal §6](Argument_Analysis_Agentic-2-formal.ipynb) invoque la sémantique
*grounded* de Dung via le solveur Tweety (JVM) comme une boîte noire : on appelle
`SimpleGroundedReasoner`, on récupère une extension. Mais **que calcule exactement ce
solveur, et pourquoi existe-t-il plusieurs sémantiques concurrentes (grounded, preferred,
stable) qui ne donnent pas toujours le même verdict ?**

Ce notebook répond à cette question en **construisant les sémantiques de zéro**, en pur
Python standard, sans JVM ni solveur externe. L'objectif n'est pas de remplacer Tweety
(moins performant, non utilisable en production sur de grands graphes) mais de **rendre
transparent** ce que le solveur fait — de façon à ce que l'appel `getModels(theory)` du
notebook 2-formal cesse d'être magique.

**Référence** : Dung, *On the Acceptability of Arguments and its Fundamental Role in
Nonmonotonic Reasoning, Logic Programming and n-Person Games* (Artificial Intelligence,
1995). Le formalisme y est introduit dans sa forme la plus abstraite, indépendante de
toute structure interne des arguments.

## 1. Cadre d'argumentation abstrait (Dung 1995)

Un **abstract argumentation framework** (AF) est un couple $F = \langle A, R \rangle$ où :

- $A$ est un ensemble fini d'**arguments** (des symboles opaques — on ne s'intéresse pas à leur contenu interne, seulement à qui attaque qui) ;
- $R \subseteq A \times A$ est une relation d'**attaque** : $(a, b) \in R$ se lit « $a$ attaque $b$ ».

L'abstraction est délibérée : la théorie de Dung porte sur la *structure du débat*
(graphe d'attaque), pas sur le sens des arguments. Deux débats aux graphes identiques
reçoivent le même verdict d'acceptabilité, quelle que soit leur matière. C'est cette
abstraction qui rend les sémantiques calculables et comparables.

Implémentons le cadre :

In [1]:
from itertools import chain, combinations


class AF:
    """Cadre d'argumentation abstrait de Dung : <args, attacks>."""

    def __init__(self, args, attacks):
        self.args = set(args)
        self.attacks = set(attacks)  # ensemble de couples (attaquant, attaque)

    def attackers(self, x):
        """Arguments qui attaquent directement x."""
        return {a for (a, b) in self.attacks if b == x}

    def __repr__(self):
        att = ", ".join(f"{a}->{b}" for (a, b) in sorted(self.attacks))
        return f"AF(args={sorted(self.args)}, attacks=[{att}])"


# Un petit exemple ludique pour démarrer : a et b s'attaquent mutuellement.
af0 = AF({"a", "b"}, {("a", "b"), ("b", "a")})
print(af0)
print("Attaquants de a :", af0.attackers("a"))

AF(args=['a', 'b'], attacks=[a->b, b->a])
Attaquants de a : {'b'}


## 2. De la conflictualité à l'admissibilité

Toute la théorie repose sur une chaîne de définitions emboîtées. Chaque notion prépare
la suivante.

### 2.1 Sans conflit interne

Un ensemble $S$ est **sans conflit** (*conflict-free*) si aucun de ses membres n'en
attaque un autre : $\nexists a, b \in S$ tels que $(a, b) \in R$.

C'est la condition minimale de cohérence : un ensemble qui se contredit n'est pas
une position défendable.

In [2]:
def is_conflict_free(af, S):
    """True si aucun argument de S n'en attaque un autre dans S."""
    S = set(S)
    for (a, b) in af.attacks:
        if a in S and b in S:
            return False
    return True


print("a,b se battent -> {a,b} sans conflit ?", is_conflict_free(af0, {"a", "b"}))
print("{a} seul  -> sans conflit ?", is_conflict_free(af0, {"a"}))

a,b se battent -> {a,b} sans conflit ? False
{a} seul  -> sans conflit ? True


### 2.2 Défait et défendu

Pour aller plus loin que la simple absence de conflit, il faut formaliser la notion de
**défense** :

- $S$ **défait** un argument $x$ (noté $S \rightsquigarrow x$) si au moins un membre de $S$
  attaque $x$ : $\exists a \in S, (a, x) \in R$.
- $S$ **défend** un argument $x$ si $S$ défait *tous* les attaquants de $x$ :
  $\forall b$ tel que $(b, x) \in R$, $S$ défait $b$.

Autrement dit : $x$ est défendu par $S$ si toute attaque contre $x$ est contrée par une
attaque venant de $S$. C'est le cœur du raisonnement argumentatif — on accepte un
argument non parce qu'il est vrai, mais parce qu'on peut le *protéger*.

In [3]:
def defeats(af, S, x):
    """True si S défait x (au moins un membre de S attaque x)."""
    return any((a, x) in af.attacks for a in S)


def defends(af, S, x):
    """True si S défend x : tout attaquant de x est défait par S."""
    return all(defeats(af, S, b) for b in af.attackers(x))


print("Dans af0, {a} défend-il a ?", defends(af0, {"a"}, "a"))
print("  (a est attaqué par b ; {a} défait-il b ? a->b :", ("a", "b") in af0.attacks, ")")

Dans af0, {a} défend-il a ? True
  (a est attaqué par b ; {a} défait-il b ? a->b : True )


### 2.3 Admissible

Un ensemble $S$ est **admissible** s'il est sans conflit *et* si chacun de ses membres
est défendu par $S$ :

$$S \text{ admissible} \iff S \text{ sans conflit} \;\land\; \forall a \in S, \; S \text{ défend } a$$

L'admissibilité capture l'idée d'une position **auto-cohérente et auto-défendable** :
aucune contradiction interne, et chaque membre résiste aux attaques extérieures.

In [4]:
def is_admissible(af, S):
    """True si S est admissible : sans conflit et chacun de ses membres est défendu."""
    S = set(S)
    if not is_conflict_free(af, S):
        return False
    return all(defends(af, S, a) for a in S)


for cand in [set(), {"a"}, {"b"}, {"a", "b"}]:
    print(f"{{{' '.join(sorted(cand)) or '∅'}}} admissible ? {is_admissible(af0, cand)}")

{∅} admissible ? True
{a} admissible ? True
{b} admissible ? True
{a b} admissible ? False


## 3. Les trois sémantiques

L'admissibilité définit un *spectre* d'ensembles acceptables. Les **sémantiques**
sélectionnent, parmi les ensembles admissibles, ceux qui méritent d'être appelés
« extensions » (le verdict final). Dung en définit trois principales, qui encodent des
**attitudes de raisonnement distinctes** :

| Sémantique | Attitude | Définition |
|------------|----------|------------|
| **Grounded** | *sceptique* (le strict minimum certain) | L'unique extension complète minimale ; obtenue par point fixe : on part de $\emptyset$ et on ajoute itérativement tout argument défendu. |
| **Preferred** | *crédule* (le maximum défendable) | Les extensions complètes **maximales** (par inclusion) ; il peut y en avoir plusieurs, incompatibles entre elles. |
| **Stable** | *auto-suffisante* (elle statut sur tout) | Une extension admissible qui **défait tout argument extérieur** : $\forall x \notin S, S \rightsquigarrow x$. |

Ces trois sémantiques ne sont pas redondantes : elles répondent à la question
« quels arguments accepter ? » sous trois régimes de prudence différents. Le théorème
central de Dung garantit l'inclusion **grounded $\subseteq$ preferred**, mais **stable
peut ne pas exister**, et **preferred peut compter plusieurs extensions incompatibles**.

Implémentons les trois par énumération (la théorie garantit que pour un AF fini, le nombre
d'extensions est fini ; l'énumération des $2^{|A|}$ sous-ensembles reste tractable pour
les petits graphes pédagogiques).

In [5]:
def powerset(args):
    """Tous les sous-ensembles de args, par taille croissante."""
    args = list(args)
    return list(chain.from_iterable(
        combinations(args, r) for r in range(len(args) + 1)
    ))


def grounded(af):
    """Extension grounded : point fixe à partir de l'ensemble vide.

    On ajoute itérativement tout argument défendu par l'ensemble courant, jusqu'à
    ce qu'aucun ajout ne soit plus possible. Le résultat est unique (théorème de Dung).
    """
    S = set()
    changed = True
    while changed:
        changed = False
        for x in af.args:
            if x not in S and defends(af, S, x):
                S.add(x)
                changed = True
    return frozenset(S)


def preferred(af):
    """Extensions preferred : ensembles admissibles maximaux par inclusion."""
    adm = [frozenset(S) for S in powerset(af.args) if is_admissible(af, S)]
    # maximal : aucun autre admissible ne le contient strictement
    return [a for a in adm if not any(a < b for b in adm)]


def stable(af):
    """Extensions stables : admissibles qui défont tout argument hors de l'extension."""
    out = []
    for S in powerset(af.args):
        S = set(S)
        if is_admissible(af, S) and all(
            defeats(af, S, x) for x in af.args if x not in S
        ):
            out.append(frozenset(S))
    return out


def resume(af, label=""):
    """Calcule et affiche les trois sémantiques d'un AF."""
    g = grounded(af)
    p = preferred(af)
    s = stable(af)
    print(f"=== {label or af} ===")
    print(f"  grounded  : {{{', '.join(sorted(g)) or '∅'}}}   (unique, sceptique)")
    print(f"  preferred : {[sorted('{' + ','.join(sorted(e)) + '}') and '{' + ','.join(sorted(e)) + '}' for e in p]}   ({len(p)} ext., crédule)")
    if s:
        print(f"  stable    : {[ '{' + ','.join(sorted(e)) + '}' for e in s]}   ({len(s)} ext.)")
    else:
        print(f"  stable    : AUCUNE (pas d'extension stable)")
    print()
    return g, p, s

## 4. Le cas canonique : grounded $\neq$ preferred $\neq$ stable

Pour que l'écart entre les trois sémantiques soit **visible**, il faut un AF assez riche
pour que les trois attitudes divergent. Considérons :

$$F = \langle \{a, b, c, d\}, \; \{(a,b), (b,a), (c,c)\} \rangle$$

- **$a$ et $b$ s'attaquent mutuellement** : ni l'un ni l'autre n'est attaqué par un tiers,
  aucun n'est défendu « gratuitement ». Ce sont des **arguments flottants** — on peut en
  accepter un (si l'on prend parti) mais pas les deux (ils se contredisent).
- **$c$ s'attaque lui-même** : un argument auto-réfuté ne peut *jamais* appartenir à un
  ensemble sans conflit (il violerait la cohérence minimale).
- **$d$ est isolé** : non attaqué, il sera accepté partout (point de référence stable).

Cet AF est **non trivial** précisément parce qu'aucune sémantique ne se réduit à une
évidence : on ne peut pas « prendre tous les non-attaqués » (ça donnerait $\{d\}$ en
ignorant les flottements) ni « tout ce qui se défend » (ça mélangerait $a$ et $b$).

In [6]:
# AF canonique : flottement a/b + auto-attaque c + isolé d
af1 = AF(
    args={"a", "b", "c", "d"},
    attacks={("a", "b"), ("b", "a"), ("c", "c")},
)
print(af1)
print()
g1, p1, s1 = resume(af1, "F = <{a,b,c,d}, {(a,b),(b,a),(c,c)}>")

AF(args=['a', 'b', 'c', 'd'], attacks=[a->b, b->a, c->c])

=== F = <{a,b,c,d}, {(a,b),(b,a),(c,c)}> ===
  grounded  : {d}   (unique, sceptique)
  preferred : ['{b,d}', '{a,d}']   (2 ext., crédule)
  stable    : AUCUNE (pas d'extension stable)



### Lecture du verdict

Les trois sémantiques donnent des résultats **différents** sur ce graphe :

| Sémantique | Résultat | Interprétation |
|------------|----------|----------------|
| **Grounded** | $\{d\}$ | Seul $d$ est *certainement* acceptable : il est le seul non attaqué. On refuse de trancher entre $a$ et $b$ (aucun n'est défendu sans prendre parti), et $c$ s'exclut de lui-même. Attitude sceptique : on n'accepte que l'incontestable. |
| **Preferred** | $\{a, d\}$ **et** $\{b, d\}$ | Deux positions maximales défendables, **incompatibles** : soit on accepte $a$ (qui défait $b$), soit $b$ (qui défait $a$). Attitude crédule : on admet chaque position auto-défendable maximale, même s'il en existe une contradictoire. |
| **Stable** | *(aucune)* | Aucune extension stable n'existe. Pourquoi ? Une extension stable devrait défaire *tout* argument extérieur. Mais $c$ n'est attaqué par personne d'autre que lui-même : aucun argument ne peut le défaire « de l'extérieur », donc aucune extension stable ne peut le statuer. Le graphe porte un argument « indécidable de l'extérieur » → la sémantique stable échoue à exister. |

C'est précisément le théorème de Dung rendu visible :
- **grounded $\subset$ preferred** ($\{d\} \subset \{a,d\}$) ;
- **preferred $\ne$ stable** (preferred existe, stable n'existe pas) ;
- la sémantique **stable n'est pas garantie d'existence** — c'est sa fragilité théorique,
  et la raison pour laquelle les solveurs pratiques (Tweety inclus) calculent en priorité
  grounded et preferred.

In [7]:
# Vérification programmatique : les trois sémantiques divergent bien.
print("grounded == {d} ?", set(g1) == {"d"})
print("preferred == {{a,d},{b,d}} ?",
      set(frozenset(e) for e in p1) == {frozenset({"a", "d"}), frozenset({"b", "d"})})
print("stable existe ?", bool(s1))
print()
print(">>> grounded ⊊ preferred :", set(g1) < set(next(iter(p1))))
print(">>> preferred existe ET stable n'existe pas (divergence) :", bool(p1) and not bool(s1))

grounded == {d} ? True
preferred == {{a,d},{b,d}} ? True
stable existe ? False

>>> grounded ⊊ preferred : True
>>> preferred existe ET stable n'existe pas (divergence) : True


## 5. Retour au solveur Tweety

Maintenant que les sémantiques sont transparentes, relisons l'appel du notebook
[2-formal §6](Argument_Analysis_Agentic-2-formal.ipynb) :

```python
reasoner_d = SimpleGroundedReasoner()
models = reasoner_d.getModels(theory)   # Collection<Extension>
```

`getModels` renvoie la collection des extensions selon la sémantique du *reasoner*. Pour
`SimpleGroundedReasoner`, c'est exactement le point fixe calculé par notre fonction
`grounded` ci-dessus — une seule extension, la plus prudente. Les *reasoners*
`SimplePreferredReasoner` et `SimpleCompleteReasoner` implémentent les autres colonnes
du tableau. Le pont JPype ne fait rien d'autre qu'appeler, en JVM, l'algorithme que nous
venons de reconstruire à la main.

**Ce que ce notebook apporte** au-delà de Tweety-5 : la compréhension de l'algorithme,
pas seulement son usage. C'est la différence entre *invoquer* un vérificateur formel
(le pipeline agentique de la série) et *comprendre* ce qu'il calcule (ce notebook).

> Pour les preuves formelles des théorèmes d'inclusion et d'existence (grounded est
> l'extension complète minimale, stable peut être vide), voir le notebook Lean
> [Tweety-5b](../Tweety/Tweety-5b-Lean-Argumentation.ipynb).

## 6. Exercices

Les trois exercices suivants approfondissent la théorie. Ils sont laissés **incomplets**
par construction (stub) : à vous de les remplir. Le notebook s'exécute de bout en bout
même non complété — vos implémentations remplaceront les `pass` / `return None`.

### Exercice 1 — Complétude vs admissibilité

Une extension **complète** est admissible *et* contient tout argument qu'elle défend :
$S$ complète $\iff$ $S$ admissible et $\forall x \in A$, si $S$ défend $x$ alors $x \in S$.
La grounded est l'unique complète minimale, les preferred sont les complètes maximales.

**Objectif** : implémenter `is_complete(af, S)` et vérifier que la grounded et les
preferred de `af1` sont bien complètes (mais qu'un admissible non-maximal comme $\{a\}$
ne l'est pas — pourquoi ?).

*Indice* : un argument défendu mais absent de $S$ suffit à le disqualifier.

In [8]:
# Exercice 1 : compléter cette fonction
def is_complete(af, S):
    """True si S est complet : admissible ET contient tout argument qu'il défend."""
    # Etape 1 : vérifier l'admissibilité (déjà disponible : is_admissible).
    # Etape 2 : pour tout x défendu par S, x doit appartenir à S.
    # TODO etudiant
    return None


# Test (à décommenter une fois complété) :
# print("grounded complet ?", is_complete(af1, set(g1)))
# print("preferred complet ?", all(is_complete(af1, set(e)) for e in p1))
# print("{a} (admissible non-maximal) complet ?", is_complete(af1, {"a"}))

### Exercice 2 — Un AF où la sémantique stable existe

Sur `af1`, la sémantique stable n'existait pas (à cause de l'auto-attaque de $c$).
Construisez un AF **sans auto-attaque ni cycle impair** où la stable existe *et* coïncide
avec la preferred.

**Objectif** : définir `af2` et vérifier que `stable(af2) == preferred(af2)` (même
collection d'extensions).

*Indice* : un AF acyclique (les arguments forment un DAG d'attaque) admet toujours une
unique extension stable, qui coïncide avec la grounded et la preferred.

In [9]:
# Exercice 2 : définir un AF où stable == preferred
af2 = AF(
    args=set(),       # TODO etudiant : choisir args
    attacks=set(),    # TODO etudiant : choisir attacks (sans auto-attaque)
)
# Test (à décommenter une fois complété) :
# g2, p2, s2 = resume(af2, "af2 (votre AF)")
# print("stable == preferred ?",
#       set(s2) == set(p2))

### Exercice 3 — Étiquetage IN / OUT / UNDEC

Les sémantiques admettent une formulation équivalente en **étiquetages** : chaque
argument reçoit l'étiquette IN (accepté), OUT (rejeté) ou UNDEC (indécidable), avec les
règles : (i) $x$ est OUT si un IN l'attaque ; (ii) $x$ est IN si tous ses attaquants sont
OUT. Le labeling grounded de `af1` devrait donner $d$=IN, $c$=OUT (auto-attaqué), et
$a, b$=UNDEC (le flottement non tranché).

**Objectif** : implémenter `grounded_labeling(af)` renvoyant un dict `{arg: "IN"/"OUT"/"UNDEC"}`,
et vérifier qu'il est cohérent avec `grounded(af)` (IN $=$ l'extension grounded, du moins
quand aucun argument n'est UNDEC dans la grounded).

*Indice* : OUT = attaqué par un IN ; IN = non attaqué, ou tous ses attaquants OUT ;
UNDEC = ni IN ni OUT (le reste). Itérez jusqu'à stabilisation, comme pour le point fixe.

In [10]:
# Exercice 3 : compléter le labeling grounded
def grounded_labeling(af):
    """Renvoie {arg: 'IN'|'OUT'|'UNDEC'} selon le labeling grounded."""
    labels = {a: "UNDEC" for a in af.args}
    # Etape 1 : IN = argument dont tous les attaquants sont OUT (ou sans attaquant).
    # Etape 2 : OUT = argument attaqué par au moins un IN.
    # Etape 3 : itérer jusqu'au point fixe (comme grounded).
    # TODO etudiant
    return labels


# Test (à décommenter une fois complété) :
# lab = grounded_labeling(af1)
# print("Labeling grounded de af1 :", lab)
# print("Cohérent avec grounded={d} ?", set(a for a, l in lab.items() if l == "IN") == set(g1))

## Conclusion

Ce notebook a reconstruit, de zéro et en Python standard, les trois sémantiques
fondatrices de l'argumentation abstraite de Dung :

- **Grounded** — l'unique extension complète minimale, attitude sceptique.
- **Preferred** — les extensions complètes maximales, attitude crédule (potentiellement
  multiples et incompatibles).
- **Stable** — les extensions admissibles qui statuent sur tout argument extérieur ;
  **non garantie d'existence**.

Le cas canonique $\langle \{a,b,c,d\}, \{(a,b),(b,a),(c,c)\} \rangle$ a montré les trois
verdicts diverger simultanément : grounded $= \{d\}$, preferred $= \{\{a,d\}, \{b,d\}\}$,
stable inexistante. C'est la richesse structurelle qui justifie l'existence de trois
sémantiques plutôt qu'une seule : aucune ne capte à elle seule toutes les nuances de
l'acceptabilité argumentative.

### Prochaines étapes

- **Utiliser le solveur** : [Tweety-5](../Tweety/Tweety-5b-Lean-Argumentation.ipynb)
  applique ces sémantiques via TweetyProject (JVM) sur des AF plus grands, où l'énumération
  exhaustive devient prohibitif. Le notebook [2-formal §6](Argument_Analysis_Agentic-2-formal.ipynb)
  en fait usage dans le pipeline agentique.
- **Prouver les théorèmes** : [Tweety-5b-Lean](../Tweety/Tweety-5b-Lean-Argumentation.ipynb)
  formalise en Lean les inclusions ($\text{grounded} \subseteq \text{preferred}$) et les
  conditions d'existence de la sémantique stable.
- **Argumentation structurée** : le formalisme abstrait de Dung ignore le contenu des
  arguments. ASPIC+, ABA et DeLP (accessibles via Tweety) reconstruisent la relation
  d'attaque à partir d'une logique sous-jacente — la série [Tweety](../Tweety/) les explore.